<a href="https://colab.research.google.com/github/cafauzi13/ESG_SentimentAnalysis/blob/main/scripts/Augmentasi_LLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## INSTALL

## SETUP GEMINI

In [26]:
# ============================================================
# INSTALL & SETUP — jalankan cell ini dulu
# ============================================================
!pip install -q google-genai scikit-learn

from google import genai
from google.genai import types
import pandas as pd
import time
from sklearn.model_selection import train_test_split

# Ganti MODEL_NAME saja
MODEL_NAME = "gemini-2.5-flash-lite"

# Test ulang
response = client.models.generate_content(
    model=MODEL_NAME,
    contents="Balas hanya dengan OK"
)
print(f"✅ Test berhasil: {response.text}")

✅ Test berhasil: OK


## LOAD DATA

In [18]:
github_url = 'https://raw.githubusercontent.com/cafauzi13/ESG_SentimentAnalysis/main/data/IndoBERT.csv'
df = pd.read_csv(github_url)
df = df.dropna(subset=['Isi Berita Clean', 'Sentiment', 'Tag'])
print(f"Total data asli: {len(df)}")
print(df['Sentiment'].value_counts())



Total data asli: 468
Sentiment
Negatif    195
Positif    172
Netral     101
Name: count, dtype: int64


## TRAIN/SPLIT

In [19]:
df_train, df_test = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['Sentiment']
)
df_test.to_csv('test_set_asli.csv', index=False)
print(f"\nTrain: {len(df_train)} | Test: {len(df_test)}")
print("✅ Test set disimpan")


Train: 374 | Test: 94
✅ Test set disimpan


## FUNGSI AUGMENTASI

In [27]:
# ============================================================
# FUNGSI AUGMENTASI — pakai client baru
# ============================================================
def augment_with_gemini(text, sentiment_label, tag_label, max_chars=800, max_retries=3):
    text_input = str(text)[:max_chars]

    prompt = f"""Kamu adalah parafrase generator untuk dataset berita ESG Indonesia.
Artikel asli (sentimen: {sentiment_label}, kategori: {tag_label}):
{text_input}

Tugas: Buat 1 parafrase artikel di atas dengan ketentuan:
- Pertahankan sentimen {sentiment_label}
- Pertahankan konteks ESG kategori {tag_label}
- Gunakan gaya bahasa berita Indonesia yang natural
- HANYA output teks parafrase, tanpa penjelasan apapun"""

    for attempt in range(max_retries):
        try:
            response = client.models.generate_content(
                model=MODEL_NAME,
                contents=prompt,
                config=types.GenerateContentConfig(
                    max_output_tokens=800,
                    temperature=0.7,
                )
            )
            hasil = response.text.strip()
            if len(hasil) > 100:
                return hasil
            else:
                print(f"    ⚠️ Output pendek ({len(hasil)} chars), retry...")
        except Exception as e:
            print(f"    ⚠️ Percobaan {attempt+1} gagal: {e}")
            time.sleep(10)

    return None

### Augmentasi pada train set

In [28]:
import time

max_count = df_train['Sentiment'].value_counts().max()
augmented_rows = []

print(f"Target per kelas: {max_count}")
print(f"Distribusi sekarang:\n{df_train['Sentiment'].value_counts()}\n")

for sentiment in df_train['Sentiment'].unique():
    df_kelas = df_train[df_train['Sentiment'] == sentiment]
    kekurangan = max_count - len(df_kelas)

    if kekurangan <= 0:
        print(f"✅ '{sentiment}': sudah cukup ({len(df_kelas)} artikel), skip.")
        continue

    print(f"\n{'='*50}")
    print(f"⏳ Sentimen: '{sentiment}' | Perlu: {kekurangan} artikel sintetis")
    print(f"{'='*50}")

    samples = df_kelas.sample(n=kekurangan, replace=True, random_state=42)
    berhasil = 0
    gagal = 0

    for i, (_, row) in enumerate(samples.iterrows()):
        print(f"  [{i+1}/{kekurangan}] Generating... ", end="", flush=True)

        try:
            teks_baru = augment_with_gemini(
                row['Isi Berita Clean'],
                row['Sentiment'],
                row['Tag']
            )

            if teks_baru and len(teks_baru) > 100:
                new_row = row.copy()
                new_row['Isi Berita Clean'] = teks_baru
                new_row['is_augmented'] = True
                augmented_rows.append(new_row)
                berhasil += 1
                print(f"✅ ({len(teks_baru)} chars)")
            else:
                gagal += 1
                print(f"⚠️ Output terlalu pendek, skip.")

        except Exception as e:
            gagal += 1
            print(f"❌ Error: {e}")

        # Simpan checkpoint setiap 10 artikel
        # supaya kalau crash di tengah, progress tidak hilang
        if (i + 1) % 10 == 0 and augmented_rows:
            df_checkpoint = pd.DataFrame(augmented_rows)
            df_checkpoint.to_csv('augmented_checkpoint.csv', index=False)
            print(f"  💾 Checkpoint disimpan ({len(augmented_rows)} artikel total)")

        # Rate limiting free tier: 15 req/menit → sleep 4.5s
        time.sleep(4.5)

    print(f"\n  Hasil '{sentiment}': {berhasil} berhasil, {gagal} gagal")

# ============================================================
# GABUNGKAN
# ============================================================
df_train['is_augmented'] = False

if augmented_rows:
    df_aug = pd.DataFrame(augmented_rows)
    df_train_final = pd.concat([df_train, df_aug], ignore_index=True)
    df_train_final = df_train_final.sample(frac=1, random_state=42).reset_index(drop=True)
    print(f"\n✅ Train set final: {len(df_train_final)} artikel")
    print(df_train_final['Sentiment'].value_counts())
    print(f"Asli: {df_train_final['is_augmented'].eq(False).sum()} | Sintetis: {df_train_final['is_augmented'].eq(True).sum()}")
    df_train_final.to_csv('train_set_augmented.csv', index=False)
    print("✅ Disimpan: train_set_augmented.csv")
else:
    print("❌ Tidak ada artikel yang berhasil diaugmentasi.")

Target per kelas: 156
Distribusi sekarang:
Sentiment
Negatif    156
Positif    137
Netral      81
Name: count, dtype: int64


⏳ Sentimen: 'Netral' | Perlu: 75 artikel sintetis
  [1/75] Generating... ✅ (645 chars)
  [2/75] Generating... ✅ (601 chars)
  [3/75] Generating... ✅ (579 chars)
  [4/75] Generating... ✅ (620 chars)
  [5/75] Generating... ✅ (868 chars)
  [6/75] Generating... ✅ (821 chars)
  [7/75] Generating... ✅ (724 chars)
  [8/75] Generating... ✅ (677 chars)
  [9/75] Generating... ✅ (484 chars)
  [10/75] Generating... ✅ (674 chars)
  💾 Checkpoint disimpan (10 artikel total)
  [11/75] Generating... ✅ (465 chars)
  [12/75] Generating... ✅ (811 chars)
  [13/75] Generating... ✅ (929 chars)
  [14/75] Generating... ✅ (506 chars)
  [15/75] Generating... ✅ (1281 chars)
  [16/75] Generating... ✅ (813 chars)
  [17/75] Generating... ✅ (456 chars)
  [18/75] Generating... ✅ (793 chars)
  [19/75] Generating... ✅ (843 chars)
  [20/75] Generating...     ⚠️ Percobaan 1 gagal: 429 RESOURCE_EXHA

KeyboardInterrupt: 

In [29]:
# Cek checkpoint yang sudah tersimpan
import pandas as pd

df_check = pd.read_csv('augmented_checkpoint.csv')
print(f"Total artikel sintetis sejauh ini: {len(df_check)}")
print(f"\nDistribusi sentimen:")
print(df_check['Sentiment'].value_counts())
print(f"\nContoh hasil augmentasi:")
print(df_check['Isi Berita Clean'].iloc[0][:300])
print("---")
print(df_check['Isi Berita Clean'].iloc[-1][:300])

Total artikel sintetis sejauh ini: 23

Distribusi sentimen:
Sentiment
Netral    23
Name: count, dtype: int64

Contoh hasil augmentasi:
PT Mentaya Sawit Mas Wilmar Central Kalimantan Project melalui Senior Conservation Forendadi Dundang gencar melakukan sosialisasi kepada masyarakat di lima desa binaannya di Kalimantan Tengah. Tujuannya adalah untuk menumbuhkan kesadaran warga dalam menjaga kawasan bernilai konservasi tinggi (HCV) y
---
Akademi Komunitas Perkebunan (AKPY) kembali menggelar pelatihan bagi 144 petani sawit dari Muara Enim dan Musi Rawas Utara di Sumatera Selatan. Pelatihan yang didukung oleh Badan Pengelola Dana Perkebunan (BPDP) dan Direktorat Jenderal Perkebunan (Ditjenbun) ini bertujuan meningkatkan kompetensi, pr
